In [0]:
# Retail Sales Lakehouse
# Data Quality Framework

from pyspark.sql import functions as F
from datetime import datetime
import uuid

DQ_TABLE = "workspace.default.data_quality_results"

dq_run_id = str(uuid.uuid4())
dq_run_time = datetime.now()

print("DQ Run ID   :", dq_run_id)
print("DQ Run Time :", dq_run_time)
print("DQ Framework initialized successfully")

In [0]:
def check_nulls(df, table_name, columns):
    results = []

    total_rows = df.count()

    for column_name in columns:
        null_count = (
            df
            .filter(F.col(column_name).isNull())
            .count()
        )

        status = "PASS" if null_count == 0 else "FAIL"

        results.append({
            "table_name": table_name,
            "rule_type": "NULL_CHECK",
            "column_name": column_name,
            "total_rows": total_rows,
            "failed_rows": null_count,
            "status": status
        })

    return results

print("Reusable NULL check function created successfully")

In [0]:
# Load Silver Customers table
silver_customers_df = spark.table(
    "workspace.default.silver_customers"
)

customer_null_results = check_nulls(
    silver_customers_df,
    "silver_customers",
    [
        "customer_id",
        "customer_name",
        "email",
        "country"
    ]
)

for result in customer_null_results:
    print(result)

In [0]:
def check_duplicates(df, table_name, key_columns):
    total_rows = df.count()

    duplicate_groups = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_count = duplicate_groups.count()

    status = "PASS" if duplicate_count == 0 else "FAIL"

    return {
        "table_name": table_name,
        "rule_type": "DUPLICATE_CHECK",
        "column_name": ",".join(key_columns),
        "total_rows": total_rows,
        "failed_rows": duplicate_count,
        "status": status
    }

print("Reusable duplicate check function created successfully")

In [0]:
customer_duplicate_result = check_duplicates(
    silver_customers_df,
    "silver_customers",
    ["customer_id"]
)

print(customer_duplicate_result)

In [0]:
def check_business_rule(df, table_name, rule_name, failure_condition):
    total_rows = df.count()

    failed_rows = (
        df
        .filter(failure_condition)
        .count()
    )

    status = "PASS" if failed_rows == 0 else "FAIL"

    return {
        "table_name": table_name,
        "rule_type": "BUSINESS_RULE",
        "rule_name": rule_name,
        "total_rows": total_rows,
        "failed_rows": failed_rows,
        "status": status
    }

print("Reusable business rule function created successfully")

In [0]:
silver_products_df = spark.table("workspace.default.silver_products")
silver_orders_df = spark.table("workspace.default.silver_orders")
silver_order_items_df = spark.table("workspace.default.silver_order_items")

business_rule_results = []

business_rule_results.append(
    check_business_rule(
        silver_products_df,
        "silver_products",
        "unit_price_must_be_positive",
        F.col("unit_price") <= 0
    )
)

business_rule_results.append(
    check_business_rule(
        silver_order_items_df,
        "silver_order_items",
        "quantity_must_be_positive",
        F.col("quantity") <= 0
    )
)

business_rule_results.append(
    check_business_rule(
        silver_orders_df,
        "silver_orders",
        "valid_order_status",
        ~F.col("order_status").isin(["Completed", "Cancelled", "Pending"])
    )
)

business_rule_results.append(
    check_business_rule(
        silver_orders_df,
        "silver_orders",
        "valid_sales_channel",
        ~F.col("sales_channel").isin(["Online", "Store"])
    )
)

for result in business_rule_results:
    print(result)

In [0]:
all_dq_results = []

# Null check results
all_dq_results.extend(customer_null_results)

# Duplicate check result
all_dq_results.append(customer_duplicate_result)

# Business rule results
all_dq_results.extend(business_rule_results)

dq_results_df = spark.createDataFrame(all_dq_results)

display(dq_results_df)

In [0]:
dq_results_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.data_quality_results")

print("Data quality results saved successfully")

In [0]:
dq_results = spark.table(
    "workspace.default.data_quality_results"
)

display(dq_results)

In [0]:
quarantine_order_items = (
    silver_order_items_df
    .filter(
        F.col("quantity").isNull() |
        (F.col("quantity") <= 0)
    )
    .withColumn("quarantine_reason", F.lit("Invalid quantity"))
    .withColumn("quarantine_time", F.current_timestamp())
)

display(quarantine_order_items)

In [0]:
quarantine_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.quarantine_order_items")

print("Quarantine table created successfully")

In [0]:
dq_summary = (
    spark.table("workspace.default.data_quality_results")
    .groupBy("status")
    .count()
)

display(dq_summary)

In [0]:
bad_order_items_data = [
    (99999, 50001, 2001, 0)   # quantity = 0 → invalid
]

bad_order_items_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity"
]

bad_order_items_df = spark.createDataFrame(
    bad_order_items_data,
    bad_order_items_columns
)

display(bad_order_items_df)

In [0]:
bad_quantity_result = check_business_rule(
    bad_order_items_df,
    "bad_order_items_test",
    "quantity_must_be_positive",
    F.col("quantity") <= 0
)

print(bad_quantity_result)

In [0]:
bad_quarantine_df = (
    bad_order_items_df
    .withColumn("quarantine_reason", F.lit("Invalid quantity"))
    .withColumn("quarantine_time", F.current_timestamp())
)

bad_quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.quarantine_order_items")

print("Bad record moved to quarantine successfully")

In [0]:
display(
    spark.table("workspace.default.quarantine_order_items")
    .orderBy(F.col("quarantine_time").desc())
)

In [0]:
failed_dq_df = spark.createDataFrame([bad_quantity_result])

failed_dq_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.data_quality_results")

print("FAIL result saved to data quality results")

In [0]:
dq_status_summary = (
    spark.table("workspace.default.data_quality_results")
    .groupBy("status")
    .count()
    .orderBy("status")
)

display(dq_status_summary)